# Datasets Experiment — Source Tier Comparison

**Question**: Does the data source tier matter for synthetic GEC benchmark quality?

Fixed variable: **CoT prompt** (winner from `prompt_engineering.ipynb`).  
Independent variable: **source dataset tier**.

| Tier | Dataset | Format | Path |
|------|---------|--------|------|
| Test set | CoNLL-2014 | `.m2` | `data/conll14/conll14st-test.m2` |
| Existing benchmark | FCE | `.m2` | `data/fce/fce.m2` |
| General (in training) | C4-200M | `.tsv` glob | `data/c4_200m/C4_200M.tsv-*` |

## Plan
1. Load & sample 50 sentences per tier
2. CoT generation on all three
3. T5 GEC correction (B → C)
4. Evaluate: **CoLA** (A/B/C), **ERRANT F0.5**, **bigram Jaccard** (memorisation proxy)
5. Comparison table

In [ ]:
import os, json, time, re, glob, warnings
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from openai import RateLimitError

warnings.filterwarnings('ignore')
load_dotenv()
if not os.getenv('OPENROUTER_API_KEY'):
    load_dotenv('Denis/.env')
assert os.getenv('OPENROUTER_API_KEY'), 'OPENROUTER_API_KEY not found'
print('Imports OK')

## 1 — Dataset Loaders

### M2 parser
CoNLL-2014 and FCE are distributed in `.m2` format — one source sentence per block,
followed by annotated edits.  
We extract `(original, ground_truth)` pairs by applying annotator-0 edits in reverse
token order to avoid index-shift bugs.

In [ ]:
def load_m2(path, annotator_id=0):
    """Parse .m2 file → DataFrame(original, ground_truth).
    Applies edits from annotator_id (default 0) in reverse token order.
    Skips blocks where the annotator made no edits (sentence already correct).
    """
    records = []
    with open(path, encoding='utf-8') as f:
        blocks = f.read().strip().split('\n\n')

    for block in blocks:
        lines = [l for l in block.strip().splitlines() if l.strip()]
        if not lines or not lines[0].startswith('S '):
            continue

        src_tokens = lines[0][2:].split()
        edits = []
        for line in lines[1:]:
            if not line.startswith('A '):
                continue
            parts = line[2:].split('|||')
            if len(parts) < 6:
                continue
            span = parts[0].split()
            start, end = int(span[0]), int(span[1])
            correction = parts[2]
            ann = int(parts[5])
            # Skip no-ops and other annotators
            if ann == annotator_id and correction != '-NONE-':
                edits.append((start, end, correction))

        if not edits:   # sentence already correct — skip
            continue

        tgt = src_tokens[:]
        for start, end, correction in sorted(edits, key=lambda x: -x[0]):
            tgt[start:end] = correction.split() if correction else []

        records.append({
            'original':     ' '.join(src_tokens),
            'ground_truth': ' '.join(tgt),
        })

    df = pd.DataFrame(records)
    print(f'  Loaded {len(df)} erroneous pairs from {path}')
    return df

In [ ]:
def sample_from_tsv(pattern, n_rows, seed=42, chunksize=5000):
    """Memory-safe sampling from TSV glob (e.g. C4-200M)."""
    files = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(f'No files matched: {pattern}')
    chunks = []
    for path in tqdm(files, desc='Sampling TSV'):
        reservoir = []
        for chunk in pd.read_csv(
            path, sep='\t', header=None,
            names=['original', 'ground_truth'], chunksize=chunksize
        ):
            reservoir.append(chunk.sample(n=min(n_rows, len(chunk)), random_state=seed))
        file_sample = pd.concat(reservoir).sample(n=n_rows, random_state=seed).reset_index(drop=True)
        chunks.append(file_sample)
        del reservoir, file_sample
    return pd.concat(chunks, ignore_index=True)

### Sample 50 sentences per tier

Same N across all three tiers ensures a fair metric comparison.
C4-200M: 5 sentences × 10 files = 50.  
CoNLL-2014 / FCE: random sample of 50 from the loaded DataFrame.

In [ ]:
N_SAMPLE = 50   # per tier
SEED     = 42

DATASETS = {
    'conll14': {'type': 'm2',  'path': 'data/conll14/conll14st-test.m2'},
    'fce':     {'type': 'm2',  'path': 'data/fce/fce.m2'},
    'c4_200m': {'type': 'tsv', 'path': 'data/c4_200m/C4_200M.tsv-*'},
}

TIER_LABELS = {
    'conll14': 'Test set',
    'fce':     'Benchmark',
    'c4_200m': 'General',
}

samples = {}
for name, cfg in DATASETS.items():
    print(f'Loading {name}...')
    if cfg['type'] == 'm2':
        df = load_m2(cfg['path'])
        samples[name] = df.sample(n=min(N_SAMPLE, len(df)), random_state=SEED).reset_index(drop=True)
    else:
        n_per_file = max(1, N_SAMPLE // len(glob.glob(cfg['path'])))
        df = sample_from_tsv(cfg['path'], n_rows=n_per_file)
        samples[name] = df.sample(n=min(N_SAMPLE, len(df)), random_state=SEED).reset_index(drop=True)
    print(f'  → {len(samples[name])} rows sampled')

print('\nSample sizes:', {k: len(v) for k, v in samples.items()})

## 2 — CoT Generation

Same CoT chain for all three tiers — this is the key experimental control.
Changing only the source data while keeping the prompt fixed isolates the effect of data tier.

In [ ]:
LLM_MODEL = 'qwen/qwen3-next-80b-a3b-instruct:free'

def make_llm(temperature=0.7):
    return ChatOpenAI(
        model=LLM_MODEL,
        api_key=os.getenv('OPENROUTER_API_KEY'),
        base_url='https://openrouter.ai/api/v1',
        temperature=temperature,
    )

COT_INSTR = (
    'You are a grammatical error generator. For each numbered sentence below:\n'
    '  Step 1: Identify the grammatical error type (e.g. subject-verb agreement, article, spelling).\n'
    '  Step 2: Write a completely NEW sentence on a different topic with the same error type.\n\n'
    'Use this exact format for each sentence:\n'
    '[N] Error type: <type>\n'
    '[N] Generated: <new sentence>\n\n'
    'Do NOT correct or rewrite the input sentence.\n'
)

cot_chain = (
    ChatPromptTemplate.from_messages([('human', COT_INSTR + '\n{batch_input}')])
    | make_llm(0.7)
    | StrOutputParser()
)
print(f'CoT chain ready — model: {LLM_MODEL}')

In [ ]:
def make_batch_input(sentences):
    return '\n'.join(f'{i+1}. {s}' for i, s in enumerate(sentences))

def parse_cot_output(text, expected):
    generated = []
    for line in text.strip().splitlines():
        m = re.match(r'^\[[0-9]+\]\s*Generated:\s*(.+)', line.strip())
        if m:
            generated.append(m.group(1).strip())
    if len(generated) == expected:
        return generated
    generated = [
        line.split('Generated:', 1)[1].strip()
        for line in text.strip().splitlines() if 'Generated:' in line
    ]
    if len(generated) == expected:
        return generated
    raise ValueError(f'CoT parse failed. Expected {expected}, got {len(generated)}:\n{text}')

def run_cot(sentences, batch_size=3, sleep=20, desc='CoT'):
    batches = [sentences[i:i+batch_size] for i in range(0, len(sentences), batch_size)]
    results = []
    for batch in tqdm(batches, desc=desc):
        batch_input = make_batch_input(batch)
        for attempt in range(7):
            try:
                out = cot_chain.invoke({'batch_input': batch_input})
                results.extend(parse_cot_output(out, len(batch)))
                break
            except ValueError:
                time.sleep(10 * (2 ** attempt))
            except RateLimitError:
                time.sleep(60 * (2 ** attempt))
        time.sleep(sleep)
    return results

print('Batch utilities ready')

In [ ]:
generated = {}
for name, df in samples.items():
    print(f'\n--- CoT generation: {name} ({TIER_LABELS[name]}) ---')
    generated[name] = run_cot(df['original'].tolist(), desc=name)
    print(f'  Generated {len(generated[name])} sentences')

print('\nGeneration complete')

In [ ]:
# Save each dataset's results to CSV (checkpoint)
for name in samples:
    out = samples[name].copy()
    out['generated'] = generated[name]
    out['tier'] = TIER_LABELS[name]
    out.to_csv(f'datasets_{name}.csv', index=False)
    print(f'Saved datasets_{name}.csv')

## 3 — T5 GEC Correction (B → C)

Correct the LLM-generated sentences with T5 to produce set C.
This lets us measure how much the model over/under-corrects generated errors.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

T5_MODEL = 'vennify/t5-base-grammar-correction'
t5_tok = AutoTokenizer.from_pretrained(T5_MODEL)
t5_mdl = AutoModelForSeq2SeqLM.from_pretrained(T5_MODEL)
t5_mdl.eval()

def correct_sentences(sentences, batch_size=8, prefix='grammar: '):
    results = []
    for i in tqdm(range(0, len(sentences), batch_size), desc='T5'):
        batch = [prefix + s for s in sentences[i:i+batch_size]]
        inputs = t5_tok(batch, return_tensors='pt', padding=True, truncation=True, max_length=128)
        with torch.no_grad():
            outputs = t5_mdl.generate(**inputs, max_new_tokens=128)
        results.extend(t5_tok.batch_decode(outputs, skip_special_tokens=True))
    return results

print('T5 loaded')

In [ ]:
corrected = {}
for name in samples:
    print(f'\nT5 correction: {name}')
    df = pd.read_csv(f'datasets_{name}.csv')
    df['t5_corrected'] = correct_sentences(df['generated'].tolist())
    df.to_csv(f'datasets_{name}.csv', index=False)
    corrected[name] = df
    print(f'  ex original:  {df["original"].iloc[0]}')
    print(f'  ex generated: {df["generated"].iloc[0]}')
    print(f'  ex t5 fixed:  {df["t5_corrected"].iloc[0]}')

## 4 — Evaluation

Three metrics, each answering a different question:

| Metric | Question |
|--------|----------|
| **CoLA** | Are generated sentences (B) grammatically erroneous? Does T5 fix them (C)? |
| **ERRANT F0.5** | Do generated error *types* match the source dataset's error distribution? |
| **Bigram Jaccard** | Is the model paraphrasing/memorising the source sentence? (memorisation proxy) |

**Expected pattern**:  
- CoLA(B) lowest for CoNLL-2014 (most controlled errors), highest for C4 general  
- ERRANT F0.5 highest for CoNLL-2014 (curated errors align with reference), lowest for C4  
- Bigram Jaccard highest for CoNLL-2014 (memorisation risk), lowest for C4

In [ ]:
from transformers import pipeline as hf_pipeline

device = 0 if torch.cuda.is_available() else -1
cola_scorer = hf_pipeline(
    'text-classification',
    model='textattack/bert-base-uncased-CoLA',
    device=device,
)

def cola_scores(texts, batch_size=32):
    scores = []
    for i in tqdm(range(0, len(texts), batch_size), desc='CoLA', leave=False):
        preds = cola_scorer(texts[i:i+batch_size], truncation=True, max_length=128)
        scores.extend(1.0 if p['label'] == 'acceptable' else 0.0 for p in preds)
    return scores

cola_results = {}
for name, df in corrected.items():
    a = cola_scores(df['original'].tolist())
    b = cola_scores(df['generated'].tolist())
    c = cola_scores(df['t5_corrected'].tolist())
    cola_results[name] = {
        'cola_A (original)':   round(np.mean(a), 3),
        'cola_B (generated)':  round(np.mean(b), 3),
        'cola_C (t5_fixed)':   round(np.mean(c), 3),
        'B < A':               bool(np.mean(b) < np.mean(a)),
        'C > B':               bool(np.mean(c) > np.mean(b)),
    }

print('CoLA scores (grammaticality rate, higher = more acceptable):')
print(pd.DataFrame(cola_results).T.to_string())

### Bigram Jaccard — Memorisation Proxy

Measures how many bigrams the generated sentence shares with its source.
High overlap = model is paraphrasing the input rather than generating new content.

This is most concerning for CoNLL-2014, which is almost certainly in the LLM's training data.
**Target**: Jaccard < 0.3 for all tiers (genuinely new sentences).

In [ ]:
def bigram_jaccard(s1, s2):
    def bigrams(s):
        toks = s.lower().split()
        return set(zip(toks, toks[1:])) if len(toks) > 1 else set()
    b1, b2 = bigrams(s1), bigrams(s2)
    if not b1 and not b2:
        return 0.0
    return len(b1 & b2) / len(b1 | b2)

jaccard_results = {}
for name, df in corrected.items():
    scores = [
        bigram_jaccard(orig, gen)
        for orig, gen in zip(df['original'], df['generated'])
    ]
    jaccard_results[name] = {
        'mean_jaccard':      round(np.mean(scores), 3),
        'pct_high_overlap':  round(np.mean([s > 0.3 for s in scores]), 3),  # % flagged
    }

print('Bigram Jaccard (source ↔ generated):')
print('  Low = model generates new content   High = possible memorisation')
print(pd.DataFrame(jaccard_results).T.to_string())

### ERRANT F0.5 — Error-Type Distribution Alignment

Computes F0.5 between the error-type distribution in the generated→T5-corrected edits (B→C)
and the reference distribution from each dataset's own source→ground_truth edits (A→gt).

This answers: *does the LLM reproduce the same kinds of errors present in the source tier?*

In [ ]:
import spacy, errant

try:
    spacy.load('en_core_web_sm')
except OSError:
    import subprocess
    subprocess.run(['python', '-m', 'spacy', 'download', 'en_core_web_sm'], check=True)

annotator = errant.load('en')

def get_edits(src, tgt):
    return annotator.annotate(annotator.parse(src), annotator.parse(tgt))

def edit_type_dist(edit_lists):
    from collections import Counter
    c = Counter(e.type for edits in edit_lists for e in edits)
    total = sum(c.values()) or 1
    return {k: v / total for k, v in c.items()}

def dist_f05(hyp, ref):
    types = set(hyp) | set(ref)
    tp = sum(min(hyp.get(t, 0), ref.get(t, 0)) for t in types)
    fp = sum(max(hyp.get(t, 0) - ref.get(t, 0), 0) for t in types)
    fn = sum(max(ref.get(t, 0) - hyp.get(t, 0), 0) for t in types)
    p  = tp / (tp + fp) if tp + fp > 0 else 0
    r  = tp / (tp + fn) if tp + fn > 0 else 0
    return round((1.25 * p * r) / (0.25 * p + r), 3) if 0.25 * p + r > 0 else 0

print('ERRANT ready')

In [ ]:
errant_results = {}
for name, df in corrected.items():
    print(f'\nERRANT: {name}')
    # Reference: source → ground_truth (each dataset's own error profile)
    ref_edits = [
        get_edits(o, g)
        for o, g in tqdm(zip(df['original'].tolist(), df['ground_truth'].tolist()),
                         total=len(df), desc='  ref', leave=False)
    ]
    # Hypothesis: generated → t5_corrected
    hyp_edits = [
        get_edits(g, t)
        for g, t in tqdm(zip(df['generated'].tolist(), df['t5_corrected'].tolist()),
                         total=len(df), desc='  hyp', leave=False)
    ]
    ref_dist = edit_type_dist(ref_edits)
    hyp_dist = edit_type_dist(hyp_edits)
    errant_results[name] = {
        'f0.5':            dist_f05(hyp_dist, ref_dist),
        'mean_ref_edits':  round(np.mean([len(e) for e in ref_edits]), 2),
        'mean_hyp_edits':  round(np.mean([len(e) for e in hyp_edits]), 2),
    }
    print(f'  F0.5 = {errant_results[name]["f0.5"]}')

print('\nERRANT F0.5 per tier:')
print(pd.DataFrame(errant_results).T.to_string())

## 5 — Comparison Table

In [ ]:
rows = []
for name in samples:
    rows.append({
        'dataset':          name,
        'tier':             TIER_LABELS[name],
        'errant_f0.5':      errant_results[name]['f0.5'],
        'cola_B':           cola_results[name]['cola_B (generated)'],
        'cola_C':           cola_results[name]['cola_C (t5_fixed)'],
        'bigram_jaccard':   jaccard_results[name]['mean_jaccard'],
        'pct_memorised':    jaccard_results[name]['pct_high_overlap'],
        'B<A':              cola_results[name]['B < A'],
        'C>B':              cola_results[name]['C > B'],
    })

comparison = pd.DataFrame(rows).set_index('dataset')
print('=== DATASET TIER COMPARISON ===')
print(comparison.to_string())
comparison.to_csv('datasets_comparison.csv')
print('\nSaved datasets_comparison.csv')